In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from os.path import getsize

In [ ]:
def get_next_analysed_game():
    # Пока не найдем игру с анализом
    while True:
        # Записываем строки игры
        buffer = []
        while True:
            line = f.readline()

            # Пустые строки не нужны
            if line == "\n":
                continue

            buffer.append(line)

            # Ходы идут последней строкой в игре
            # и всегда начинаются с "1."
            if line.startswith("1."):
                break
                
        # Если есть анализ - берем
        if ("%eval" in line):
            return buffer

In [ ]:
def params_to_dict(str_list):
    """
    Принимает строки в виде
    [key "value"]
    И переводит их в словарь
    https://en.wikipedia.org/wiki/Portable_Game_Notation
    """
    return {
        a: b.strip('"') 
        for a, b in [
            i.strip("\n").strip("[]").split(" ", 1) 
            for i in str_list
        ]
    }

In [ ]:
PGN_FILE = "pgn/lichess_db_standard_rated_2024-01.pgn"
f = open(PGN_FILE, mode="r")

In [ ]:
i = 0
elos = []
moves = []

while i < 10_000_000:
    if i%1000 == 0:
        print(i, end="\r")
    
    g = get_next_analysed_game()
    game = params_to_dict(g[:-1])
    
    time = game["TimeControl"].split("+")[0]
    if time not in ["600", "900"]:
        continue
    if abs(int(game.get("WhiteRatingDiff", 1000))) > 15 or abs(int(game.get("WhiteRatingDiff", 1000))) > 15:
        continue
    
    s = g[-1].replace("[", "").replace("]", "")
    s = s.replace("!", "").replace("?", "").replace("+", "").replace("#", "")
    s = s.split(" ")
    s = s[:-1]
    s = s[1::8]
    #s = " ".join(s)
    
    elo = (
        int(game["WhiteElo"]) + 
        int(game["BlackElo"])
    ) // 2
    
    elos.append(elo)
    moves.append(s)
    
    i += 1

In [ ]:
n_games = len(elos)
n_games

In [ ]:
line_count = {}
line_rating = {}

for elo, move in zip(elos, moves):
    
    for i in range(1, min(20, len(move))):
        
        key = " ".join(move[:i])
        line_count[key] = line_count.get(key, 0) + 1
        line_rating[key] = line_rating.get(key, 0) + elo

In [ ]:
result = {
    k: v / line_count[k]
    for k, v in line_rating.items()
    if line_count[k] > 100
}

In [ ]:
series = pd.Series(result).sort_values()

In [ ]:
px.line(series.values, template="plotly_white")

In [ ]:
# !!
book_moves = (
    series.index
    .str.split(" ").map(len)
    .map(lambda x: x if x < 12 else 12)
)
px.line(
    series.groupby(book_moves).mean(),
    template="plotly_white"
)

In [ ]:
series.sort_index().to_dict()

In [ ]:
tree = {}
for key, value in series.sort_index().to_dict().items():
    current_node = tree
    for move in key.split(" "):
        if not (move in current_node):
            current_node[move] = {}
        current_node = current_node[move]
    current_node["mean"] = value

In [ ]:
new_moves = "Nf3 Nf6 d4 d5 h9".split(" ")
current_node = tree
for move in new_moves:
    if len(current_node) == 1:
        print(move)
        print(current_node)
        break
    current_node = current_node[move]